In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime



In [ ]:
# Data Analytics Approach: Finding Fraud Patterns & Business Rules

In [19]:
# Data Loading

df = pd.read_csv(r"D:\data_analytics\projects\upi fraud detection\upi_transactions_synthetic.csv")

df["transaction_datetime"] = pd.to_datetime(df["transaction_datetime"])


In [20]:
print(f"Data loaded: {df.shape[0]:,} transactions and {df.shape[1]} columns")

print(f"Fraud rate: {(df["is_fraud"].mean()*100 ):.2f}%  ({df["is_fraud"].sum():,} fraud transactions)")

Data loaded: 80,000 transactions and 18 columns
Fraud rate: 10.05%  (8,040 fraud transactions)


In [21]:
# 1. BASIC PATTERN SETUP

# Time Pattern

df['transaction_hour'] = df['transaction_datetime'].dt.hour
df['transaction_day'] = df['transaction_datetime'].dt.day_name()
df['transaction_month'] = df['transaction_datetime'].dt.month_name()
df['is_weekend'] = df['transaction_datetime'].dt.dayofweek.isin([5, 6]).astype(int)
df['is_night'] = ((df['transaction_hour'] >= 1) & (df['transaction_hour'] <= 5)).astype(int) # peak fraud transaction hours
df['is_business_hours'] = ((df['transaction_hour'] >= 9) & (df['transaction_hour'] <= 17)).astype(int) # business hours (9 AM to 5 PM)

In [22]:
# Location Pattern

df["location_mismatch"] = (df["location"] != df["merchant_city"]).astype(int)
# we are checking if user city and merchant city matches or not, because if they are not same 
# it's a matter or suspicion. (because why are we paying someone in another city)

In [23]:
# user behaviour patterns

df = df.sort_values(["user_id", "transaction_datetime"])

df["time_since_last_txn_minutes"] = df.groupby("user_id")["transaction_datetime"].diff().dt.total_seconds()/60

df["rapid_transactions"] = (df["time_since_last_txn_minutes"] < 60).astype(int)


df["rapid_transactions"].value_counts()

rapid_transactions
0    70855
1     9145
Name: count, dtype: int64

In [ ]:
# User tarnsaction count

user_txn_count = df.groupby("user_id").size().reset_index(name = "user_total_txns")
#df = pd.merge(df, user_txn_count, on='user_id', how='left') 
# [we first created a dataframe by grouping the user_id and counting the number of transaction 
# made by each user and then added that column  in the main dataframe, so that the total no. of 
# transaction shows in every cell where the user_id appears]

df[["user_id", "user_total_txns"]].head(5)


,user_id,user_total_txns
0,U00001,35
1,U00001,35
2,U00001,35
3,U00001,35
4,U00001,35


In [ ]:
# FRAUD PATTERN DISCOVERY

In [26]:
def analyze_pattern(pattern_name, condition):
    pattern_data = df[condition].copy()
    total_in_pattern = len(pattern_data)

    if total_in_pattern == 0:
        return None
    
    fraud_in_pattern = pattern_data["is_fraud"].sum()
    fraud_rate = (fraud_in_pattern/total_in_pattern)*100
    avg_amount = pattern_data["amount"].mean()

    return{
        "pattern": pattern_name,
        "total_transaction": total_in_pattern,
        "fraud_transactions": fraud_in_pattern,
        "fraud_rate": fraud_rate,
        "avg_amount": avg_amount,
        "percentage_of_fraud": (fraud_in_pattern/df["is_fraud"].sum())*100
    }

In [31]:
# DEFINING PATTERNS TO ANALYZE

#(okay so we first created a function (analyze_pattern) and then we created patterns. to automate the pattern analysis
# one by one, we created the list or else we would had to create multiple variable eg: p1 to save the first pattern 
# results, then p2 for the next one... to apply the functio on multiple patterns , but like this we can apply the functio once, and it will check every 
# patterna and save the results of every pattern in a single list)
patterns = []

# pattern 1: Failed Transaction
patterns.append(analyze_pattern(
    "FAILED TRANSACTION",
    (df["transaction_status"]=="Failed")
))

# pattern 2: Medium Large Amount ( >5k-<1500)
patterns.append(analyze_pattern(
    "MEDIUM LARGE AMOUNT(>5,000-<15,000)",
    (df["amount"]>5000) & (df['amount'] <= 15000)
))

# pattern 3: Night Heist
patterns.append(analyze_pattern(
    "LATE NIGHT LARGE TRANSACTION (1-5 Am + >5,000)",
    (df["amount"] > 5000) & (df["is_night"])
))

# pattern 4: Location mismatch + Medium amount
patterns.append(analyze_pattern(
    "LOCATION MISMATCH + >3,000",
    (df["location_mismatch"]==1) & (df["amount"]>3000)
))

# pattern 5: Weekend Large Transaction
patterns.append(analyze_pattern(
    "WEEKEND LARGE TRANSACTION",
    (df["is_weekend"]==1) & (df["amount"]> 5000)
))

#pattern 6: Huge Transaction (>15k)
patterns.append(analyze_pattern(
    "HUGE AMOUNT TRANSACTION (>15,000)",
    (df["amount"]>15000)
))

# pattern 7 : small transaction (<1k)
patterns.append(analyze_pattern(
    "SMALL TRANSACTIONS (< 1,000)",
    (df["amount"]< 1000)
))

# pattern 8: Business hour transactions
patterns.append(analyze_pattern(
    "BUSINESS HOURS TRANSACTIONS",
    (df["is_business_hours"]==1) & (df["amount"]> 5000)
))


In [32]:
# Remove none values 
# ( if the function returned none after being applied it will removed from the list)
patterns = [ p for p in patterns if p is not None]

# creating patterns in dataframe
patterns_df = pd.DataFrame(patterns)
patterns_df = patterns_df.sort_values("fraud_rate", ascending = False)

# presenting the patterns
print("FRAUD PATTERNS DISCOVERED")
print("-"*80)
for idx, row in patterns_df.iterrows():
    print(f"\nPattern: {row["pattern"]}:")
    print(f"\nFraud Rate:         {row["fraud_rate"]:.1f}%")
    print(f"Transactions:       {row["total_transaction"]:,}")
    print(f"Fraud Transcations: {row["fraud_transactions"]:,}")
    print(f"Avg Amount:        ₹{row["avg_amount"]:,.0f}")
    print(f"Covers:             {row["percentage_of_fraud"]:.1f}% of all fraud")


FRAUD PATTERNS DISCOVERED
--------------------------------------------------------------------------------

Pattern: LATE NIGHT LARGE TRANSACTION (1-5 Am + >5,000):

Fraud Rate:         78.0%
Transactions:       3,170
Fraud Transcations: 2,474
Avg Amount:        ₹23,027
Covers:             30.8% of all fraud

Pattern: LOCATION MISMATCH + >3,000:

Fraud Rate:         72.7%
Transactions:       3,156
Fraud Transcations: 2,295
Avg Amount:        ₹17,476
Covers:             28.5% of all fraud

Pattern: HUGE AMOUNT TRANSACTION (>15,000):

Fraud Rate:         71.0%
Transactions:       7,295
Fraud Transcations: 5,181
Avg Amount:        ₹25,970
Covers:             64.4% of all fraud

Pattern: WEEKEND LARGE TRANSACTION:

Fraud Rate:         67.6%
Transactions:       3,101
Fraud Transcations: 2,095
Avg Amount:        ₹20,720
Covers:             26.1% of all fraud

Pattern: MEDIUM LARGE AMOUNT(>5,000-<15,000):

Fraud Rate:         62.1%
Transactions:       3,621
Fraud Transcations: 2,250
Avg Amoun

In [ ]:
# BUSINESS RULES

print("="*70)
print("BUSINESS RULES SUGGESTED")
print("="*70)

# HIGH RISK RULES 
print("\nHIGH-RISK RULES (BLOCK IMMEDIATELY IF >60 fraud):")
print("-"*70)

high_risk = patterns_df[patterns_df["fraud_rate"]>60]

for idx, row in high_risk.iterrows():
    print(f"\n IF {row["pattern"]}")
    print(f"   Action: BLOCK transaction")
    print(f"   Reason: {row["fraud_rate"]:.1f}% fraud rate")
    print(f"   Verification steps:")
    print(f"     1. Immediate phone call to customer")
    print(f"     2. Biometric verification (fingerprint/face ID)")
    print(f"     3. Security questions only customer would know")
    print(f"     4. If verified → APPROVE with monitoring")
    print(f"     5. If suspicious → HOLD and investigate")
    


# MEDIUM RISK RULES
print(f"\nMedium-RISK RULES (BLOCK IMMEDIATELY IF >60 fraud):")
print("-"*70)

medium_risk = patterns_df[(patterns_df['fraud_rate'] > 5) & (patterns_df['fraud_rate'] <= 60)]
for idx, row in medium_risk.iterrows():
    print(f"\n IF {row['pattern']}")
    print(f"   Action: Require OTP + Security Question")
    print(f"   Reason: {row['fraud_rate']:.1f}% fraud rate")
    print(f"   User impact: Extra 30 seconds for {row['total_transaction']:,} transactions")


# # LOW RISK RULES
print("\nLOW-RISK RULES (AUTO-APPROVE - <5% fraud):")
print("-"*60)

low_risk = patterns_df[patterns_df['fraud_rate'] <= 5]
for idx, row in low_risk.iterrows():
    print(f"\n IF {row['pattern']}")
    print(f"   Action: Auto-approve instantly")
    print(f"   Reason: Only {row['fraud_rate']:.1f}% fraud rate")
    print(f"   Benefit: Fast approval for {row['total_transaction']:,} transactions")


BUSINESS RULES SUGGESTED

HIGH-RISK RULES (BLOCK IMMEDIATELY IF >60 fraud):
----------------------------------------------------------------------

 IF LATE NIGHT LARGE TRANSACTION (1-5 Am + >5,000)
   Action: BLOCK transaction
   Reason: 78.0% fraud rate
   Verification steps:
     1. Immediate phone call to customer
     2. Biometric verification (fingerprint/face ID)
     3. Security questions only customer would know
     4. If verified → APPROVE with monitoring
     5. If suspicious → HOLD and investigate

 IF LOCATION MISMATCH + >3,000
   Action: BLOCK transaction
   Reason: 72.7% fraud rate
   Verification steps:
     1. Immediate phone call to customer
     2. Biometric verification (fingerprint/face ID)
     3. Security questions only customer would know
     4. If verified → APPROVE with monitoring
     5. If suspicious → HOLD and investigate

 IF HUGE AMOUNT TRANSACTION (>15,000)
   Action: BLOCK transaction
   Reason: 71.0% fraud rate
   Verification steps:
     1. Immediat

In [39]:
# RISK SCORING SYSTEM  (scores are found by multiplying teh fraud rate with 0.7)

def calculated_risk_score(transaction):
    score = 0
    reasons = []

    if transaction.get("amount") > 15000:                   
        score+=50
        reasons.append("Huge amount Transaction (>15k): 71% fraud rate")
    elif 5000 < transaction.get("amount") <= 15000:          
        score += 43
        reasons.append("Large amount Transaction (>5k-<15k): 62% fraud rate")
    elif transaction.get("amount") < 1000:                    
        score += 0
        reasons.append("Small Amount Transactions (<1k): 0% fraud rate")
    if transaction.get("transaction_status") == "Failed":
        score += 38
        reasons.append("Failed Transaction: 54% fraud rate")
    if transaction.get("is_night") == 1 and transaction.get("amount") > 500:
        score += 55
        reasons.append("Large Night Transaction: 78% fraud rate")
    if transaction.get("location_mismatch") == 1 and transaction.get("amount")>3000:
        score += 51
        reasons.append("Transaction > 3k & Location mismatch: 72.7% fraud rate")
    if transaction.get("is_weekend") == 1 and transaction.get("amount")>5000:
        score += 47
        reasons.append("Transaction > 5k on Weekends: 64.6% fraud rate")
    if transaction.get("is_business_hours")==1 and transaction.get("amount")>5000:
        score += 43
        reasons.append("Transaction > 5k during Business hours: 62.1% fraud rate")

    return min(max(score, 0),100), reasons

# min(max(score, 0),100) 
# max(score, 0) - Ensures score is never below 0  (If score = -15, becomes 0, If score = 25, stays 25)
# min(..., 100) - Ensures score is never above 100  (If result from step 1 is 125 → becomes 100, If result from step 1 is 75 → stays 75)
# even even teh scores exceed 100 the results will show 100 along with all the listed reasons


In [57]:
# values to test the pattern

test_cases = [
    
    {
        'amount': 500,
        'is_night': 0,
        'location_mismatch': 0,
        'is_weekend': 0,
        'is_business_hours': 0,
        'transaction_status': 'Success'
    },
    {
        'amount': 8000,
        'is_night': 0,
        'location_mismatch': 1,
        'is_weekend': 0,
        'is_business_hours': 0,
        'transaction_status': 'Failed'
    },
    {
        'amount': 4000,        
        'is_night': 0,
        'location_mismatch': 0,
        'is_weekend': 0,
        'is_business_hours': 0,
        'transaction_status': 'Failed'  
    },
    {
        'amount': 7000,       
        'is_night': 0,
        'location_mismatch': 0,
        'is_weekend': 0,
        'is_business_hours': 1, 
        'transaction_status': 'Success'
    },
    {
        'amount': 3500,        
        'is_night': 0,
        'location_mismatch': 1,  
        'is_weekend': 0,
        'is_business_hours': 0,
        'transaction_status': 'Success'
    },  
    {
        'amount': 8000,        
        'is_night': 0,
        'location_mismatch': 0,
        'is_weekend': 1,      
        'is_business_hours': 1,
        'transaction_status': 'Success'
    }
    
]

for i, test in enumerate(test_cases,1):
    score, reasons = calculated_risk_score(test)

    if score >= 50:
        action = "ESCALATE TO FRAUD TEAM"
    elif score >=43:
        action = "STRONG VERIFICATION (OTP + Biometric)"
    elif score >= 20:
        action = "FLAG FOR REVIEW"
    else:
        action = "AUTO APPROVE"

    print(f"\nTest Case {i}: Score = {score}/100")
    print(f"Actions: {action}")
    print(f"Reasons: {", ".join(reasons[:3])}")



Test Case 1: Score = 0/100
Actions: AUTO APPROVE
Reasons: Small Amount Transactions (<1k): 0% fraud rate

Test Case 2: Score = 100/100
Actions: ESCALATE TO FRAUD TEAM
Reasons: Large amount Transaction (>5k-<15k): 62% fraud rate, Failed Transaction: 54% fraud rate, Transaction > 3k & Location mismatch: 72.7% fraud rate

Test Case 3: Score = 38/100
Actions: FLAG FOR REVIEW
Reasons: Failed Transaction: 54% fraud rate

Test Case 4: Score = 86/100
Actions: ESCALATE TO FRAUD TEAM
Reasons: Large amount Transaction (>5k-<15k): 62% fraud rate, Transaction > 5k during Business hours: 62.1% fraud rate

Test Case 5: Score = 51/100
Actions: ESCALATE TO FRAUD TEAM
Reasons: Transaction > 3k & Location mismatch: 72.7% fraud rate

Test Case 6: Score = 100/100
Actions: ESCALATE TO FRAUD TEAM
Reasons: Large amount Transaction (>5k-<15k): 62% fraud rate, Transaction > 5k on Weekends: 64.6% fraud rate, Transaction > 5k during Business hours: 62.1% fraud rate
